# 01 — Pourquoi fine-tuner ? Du mini-GPT à Mistral 7B

**Chapitre 3 · Fine-tuning d'un grand modèle** · durée conseillée : 20 min · CPU uniquement · [![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/fr/chapitre_3/01_pourquoi_finetuner.ipynb)

![Parcours du chapitre 3](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/fr/chapitre_3/figures/00_parcours.png)

Au chapitre 2, nous avons **entraîné de zéro** un mini-GPT de 200 000 paramètres sur un seul corpus. Il a appris l'orthographe, la mise en page, quelques mots. Mais il ne « sait » rien du monde, et sur Kaamelott (4 000 caractères) il a surtout récité.

Les vrais modèles de langage sont entraînés une première fois sur des **milliers de milliards de tokens** (pré-entraînement), ce qui coûte des millions d'euros. Personne ne refait cette étape pour adapter un modèle à son usage : on part d'un modèle pré-entraîné et on le **fine-tune**, c'est-à-dire qu'on **continue l'entraînement** sur nos données.

**Le fil rouge du chapitre :** comment transformer un modèle généraliste de 7 milliards de paramètres en un modèle qui imite les dialogues de *Kaamelott*, de Platon ou de Shakespeare, **sans le détruire**, avec un seul GPU ?

**À la fin de ce notebook, vous saurez :**
1. situer Mistral 7B par rapport au mini-GPT du chapitre 2 : **même architecture**, autre échelle ;
2. expliquer ce que change un tokeniseur **BPE** par rapport au tokeniseur caractère ;
3. choisir entre *prompting*, RAG et fine-tuning ;
4. écrire la loss du **supervised fine-tuning** (SFT) : la cross-entropy du chapitre 2, **masquée** sur la réponse ;
5. distinguer trois objectifs : adaptation de domaine, style conversationnel, instruction.

In [1]:
# Préparation. Sur Colab : on récupère le dépôt du cours (données + modules partagés).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("finetune.py"):
    !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
    %cd ai-masterclass/fr/chapitre_3
sys.path.insert(0, "../chapitre_2")   # minigpt.py du chapitre 2

import math, random, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # avertissement tqdm sans importance
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

import finetune as ft   # fichier finetune.py, à côté de ce notebook
torch.manual_seed(1337)
print("PyTorch", torch.__version__)
import minigpt   # le mini-GPT du chapitre 2
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"   # téléchargement : seulement config + tokeniseur (quelques Mo)

PyTorch 2.15.0+cu132


## 1. Même architecture, autre échelle

Mistral 7B est un **decoder-only**, exactement comme notre `GPTLanguageModel` du notebook 08 du chapitre 2. Pour le vérifier sans télécharger 14 Go de poids, on construit le modèle sur le périphérique **`meta`** : PyTorch crée la structure et les formes des tenseurs, mais **n'alloue aucune mémoire** pour les valeurs.

In [2]:
config = AutoConfig.from_pretrained(MODEL_ID)
with torch.device("meta"):                      # structure seulement, 0 octet de poids
    mistral = AutoModelForCausalLM.from_config(config)

n_mistral = sum(p.numel() for p in mistral.parameters())
mini = minigpt.GPTLanguageModel(vocab_size=65, block_size=64, n_embd=64, n_head=4, n_layer=4)
n_mini = sum(p.numel() for p in mini.parameters())
print(f"mini-GPT (chapitre 2) : {n_mini:>15,} paramètres")
print(f"Mistral 7B            : {n_mistral:>15,} paramètres  (x {n_mistral / n_mini:,.0f})")
print()
print(mistral)

mini-GPT (chapitre 2) :         211,777 paramètres
Mistral 7B            :   7,248,023,552 paramètres  (x 34,225)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
    

### Lire l'affichage avec les yeux du chapitre 2

| Chapitre 2 (`minigpt.py`) | Mistral 7B | Ce qui change |
|---|---|---|
| `token_embedding_table` (65 × 64) | `embed_tokens` (32 768 × 4 096) | vocabulaire de **sous-mots** (BPE, §2) |
| `position_embedding_table` | — | positions encodées par **rotation** de q et k (RoPE), dans l'attention |
| `Head.query / key / value` | `q_proj / k_proj / v_proj` | toutes les têtes dans **une seule** matrice ; 32 têtes de requêtes mais seulement 8 têtes clé/valeur partagées (*grouped-query attention*), d'où `k_proj` en 4 096 → 1 024 |
| `MultiHeadAttention.proj` | `o_proj` | identique |
| `FeedForward` : C → 4C → ReLU → C | `gate_proj`, `up_proj`, `down_proj` | variante **SwiGLU** : 4 096 → 14 336 → 4 096 |
| `LayerNorm` | `RMSNorm` | normalisation sans recentrage |
| `Block` × 4 | `MistralDecoderLayer` × 32 | 8 fois plus profond |
| `lm_head` | `lm_head` | identique |
| `block_size = 64` caractères | 32 768 tokens | contexte ~ 500 fois plus long |

**Le problème reste le même** : prédire le token suivant, avec une cross-entropy. Tout ce que vous avez construit au chapitre 2 est dans ce modèle ; seules l'échelle et quelques briques modernisées changent.

Où sont les 7 milliards ? Faisons le compte à la main avec les dimensions du `config.json`.

In [3]:
print({k: config.to_dict()[k] for k in ft.MISTRAL_7B})
assert all(config.to_dict()[k] == v for k, v in ft.MISTRAL_7B.items())

comptes = ft.parametres_mistral(ft.MISTRAL_7B)
for nom, n in comptes.items():
    print(f"{nom:20s} {n:>15,}  ({n / n_mistral:6.1%})")
print(f"{'total':20s} {sum(comptes.values()):>15,}   <- identique au décompte PyTorch : {sum(comptes.values()) == n_mistral}")
print()
for nom, (d_in, d_out) in ft.formes_lineaires().items():
    print(f"  {nom:10s} {d_in:>6} -> {d_out:<6} : {d_in * d_out:>11,} paramètres par couche")

{'vocab_size': 32768, 'hidden_size': 4096, 'intermediate_size': 14336, 'num_hidden_layers': 32, 'num_attention_heads': 32, 'num_key_value_heads': 8}
embedding                134,217,728  (  1.9%)
blocs (linéaires)      6,979,321,856  ( 96.3%)
normes                       266,240  (  0.0%)
lm_head                  134,217,728  (  1.9%)
total                  7,248,023,552   <- identique au décompte PyTorch : True

  q_proj       4096 -> 4096   :  16,777,216 paramètres par couche
  k_proj       4096 -> 1024   :   4,194,304 paramètres par couche
  v_proj       4096 -> 1024   :   4,194,304 paramètres par couche
  o_proj       4096 -> 4096   :  16,777,216 paramètres par couche
  gate_proj    4096 -> 14336  :  58,720,256 paramètres par couche
  up_proj      4096 -> 14336  :  58,720,256 paramètres par couche
  down_proj   14336 -> 4096   :  58,720,256 paramètres par couche


**96 % des paramètres** sont dans les couches linéaires des 32 blocs, et les deux tiers dans les FeedForward. C'est là que LoRA (notebook 03) et la quantification 4 bits (notebook 04) vont agir.

---
## 2. Le tokeniseur : des sous-mots au lieu des caractères

Au chapitre 2, un token = un caractère (vocabulaire de 65 à ~100 symboles). Mistral utilise un tokeniseur **BPE** (*byte-pair encoding*) de 32 768 tokens : les suites de caractères fréquentes deviennent un seul token.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("taille du vocabulaire :", len(tokenizer))

phrases = {
    "kaamelott": "PERCEVAL: Merde j’ai plus d’pierre ! Qu’est c’qu’on fait ?",
    "platon": "SOCRATE: Eh bien donc, dis à nos juges qui est-ce qui est capable de rendre les jeunes gens meilleurs.",
    "shakespeare": "First Citizen: Before we proceed any further, hear me speak.",
}
for nom, p in phrases.items():
    toks = tokenizer.tokenize(p)
    print(f"\n{nom} : {len(p)} caractères -> {len(toks)} tokens")
    print(toks)

taille du vocabulaire : 32768

kaamelott : 58 caractères -> 25 tokens
['▁PER', 'CE', 'VAL', ':', '▁Mer', 'de', '▁j', '’', 'ai', '▁plus', '▁d', '’', 'p', 'ierre', '▁!', '▁Qu', '’', 'est', '▁c', '’', 'qu', '’', 'on', '▁fait', '▁?']

platon : 102 caractères -> 32 tokens
['▁SO', 'CR', 'ATE', ':', '▁E', 'h', '▁bien', '▁donc', ',', '▁dis', '▁à', '▁nos', '▁jug', 'es', '▁qui', '▁est', '-', 'ce', '▁qui', '▁est', '▁capable', '▁de', '▁rend', 're', '▁les', '▁je', 'unes', '▁g', 'ens', '▁meille', 'urs', '.']

shakespeare : 60 caractères -> 14 tokens
['▁First', '▁Cit', 'izen', ':', '▁Before', '▁we', '▁proceed', '▁any', '▁further', ',', '▁hear', '▁me', '▁speak', '.']


- `▁` marque une espace devant le mot : les espaces font partie des tokens.
- Les mots fréquents (`▁Before`, `▁dialogue`, `▁capable`) sont **un seul token** ; les mots moins fréquents (`▁jug|es`), les élisions argotiques (`▁d|’|p|ierre`) et les noms en majuscules (`▁SO|CR|ATE`) sont découpés en morceaux.
- L'anglais est un peu plus « compact » que le français : le tokeniseur a vu plus d'anglais pendant sa construction.

Mesurons-le sur un extrait de chaque corpus : combien de caractères par token, en moyenne ?

In [5]:
print(f"{'corpus':12s} {'caractères':>10s} {'tokens BPE':>10s} {'car/token':>10s}")
for nom in ["kaamelott", "platon", "shakespeare"]:
    extrait = ft.charger_texte(nom)[:20_000]
    n_tok = len(tokenizer(extrait).input_ids)
    print(f"{nom:12s} {len(extrait):10,} {n_tok:10,} {len(extrait) / n_tok:10.2f}")

corpus       caractères tokens BPE  car/token
kaamelott         4,237      1,545       2.74
platon           20,000      6,902       2.90
shakespeare      20,000      6,430       3.11


Un token ≈ 3 caractères (moins pour l'argot de Kaamelott). Conséquence pratique : un contexte de 512 tokens ≈ 1 500 caractères, soit **plus de 20 fois** le `block_size` de notre mini-GPT. Et chaque token porte déjà du sens : l'embedding de `▁capable` a été appris sur des milliards de phrases.

> **Le piège de l'apostrophe, version BPE.** Au chapitre 2, le tokeniseur Kaamelott ne connaissait pas `'`. Ici, `'` et `’` existent tous les deux : rien n'est impossible à écrire, mais ce ne sont **pas les mêmes tokens**. Le modèle devra apprendre que Kaamelott utilise `’`.

---
## 3. Trois façons d'adapter un modèle

| Méthode | Ce qu'on change | Quand l'utiliser | Limite |
|---|---|---|---|
| **Prompting** (consigne, exemples dans le prompt) | rien, juste l'entrée | tester vite, peu d'exemples | le style s'efface vite, contexte limité, coût à chaque appel |
| **RAG** (on colle des documents pertinents dans le prompt) | rien, on ajoute une recherche documentaire | **connaissances** à jour, citer ses sources | n'apprend pas un style ou un format |
| **Fine-tuning** | les **poids** | apprendre un **style**, un format, un comportement, une tâche | données à préparer, GPU, risque d'**oubli** |

Notre objectif — parler comme Perceval, comme Socrate dans la traduction de Cousin ou comme un personnage de Shakespeare — est un **style** : c'est un cas d'école pour le fine-tuning. Pour faire répondre un modèle à des questions sur le règlement de votre université, le RAG serait le bon outil.

---
## 4. La loss du fine-tuning supervisé (SFT)

![Masque de la loss](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/fr/chapitre_3/figures/01_masque_loss.png)

Un exemple de SFT est un couple **(prompt x, réponse y)**. On concatène les deux, on fait passer la séquence dans le modèle, et on calcule la **même cross-entropy qu'au chapitre 2**… mais **uniquement sur les tokens de la réponse** :

$$\mathcal{L}(\theta) = -\sum_{t=1}^{T} \log p_\theta(y_t \mid y_{<t}, x)$$

Pourquoi ne pas compter le prompt ? Parce qu'on ne veut pas apprendre au modèle à **écrire des consignes** ou à recopier le contexte : on veut qu'il apprenne **à répondre**.

En PyTorch, le masque est un simple **label `-100`** : `F.cross_entropy` ignore ces positions (`ignore_index=-100` par défaut).

In [6]:
torch.manual_seed(0)
V, T = 10, 8
logits = torch.randn(1, T, V)                                    # sortie du modèle (B=1, T=8, V=10)
labels = torch.tensor([[-100, -100, -100, -100, 3, 7, 1, 2]])   # 4 tokens de prompt masqués, 4 de réponse

loss_masquee = F.cross_entropy(logits.view(-1, V), labels.view(-1))      # ignore_index=-100 par défaut
a_la_main = F.cross_entropy(logits[0, 4:], labels[0, 4:])                # seulement les 4 dernières positions
print(f"loss avec masque -100     : {loss_masquee:.4f}")
print(f"loss sur la réponse seule : {a_la_main:.4f}")
assert torch.allclose(loss_masquee, a_la_main)

loss avec masque -100     : 3.6602
loss sur la réponse seule : 3.6602


### Sur un vrai exemple Kaamelott

Les modèles « Instruct » sont entraînés avec un **format de conversation** (*chat template*). Pour Mistral : `<s>[INST] message de l'utilisateur [/INST] réponse</s>`. Construisons un exemple et regardons quels tokens comptent dans la loss.

In [7]:
prompt = [{"role": "user", "content": "Continue ce dialogue de Kaamelott avec la réplique suivante.\n\n"
                                      "BOHORT: La femelle lièvre, c’est la hase."}]
reponse = [{"role": "assistant", "content": "KARADOC: Ah non, moi je connais que le cri."}]

ids_prompt = tokenizer.apply_chat_template(prompt, add_generation_prompt=True)["input_ids"]
ids_complet = tokenizer.apply_chat_template(prompt + reponse)["input_ids"]
assert ids_complet[:len(ids_prompt)] == ids_prompt      # le prompt est un PRÉFIXE de l'exemple complet

labels = [-100] * len(ids_prompt) + ids_complet[len(ids_prompt):]
print(f"{len(ids_complet)} tokens dont {len(ids_complet) - len(ids_prompt)} comptent dans la loss\n")
for i, l in zip(ids_complet, labels):
    marque = "   " if l == -100 else " ✔ "
    print(f"{marque}{tokenizer.convert_ids_to_tokens(i)!r}", end="  " if l == -100 else "\n")

59 tokens dont 19 comptent dans la loss

   '<s>'     '[INST]'     '▁Contin'     'ue'     '▁ce'     '▁dialogue'     '▁de'     '▁Ka'     'am'     'el'     'ott'     '▁avec'     '▁la'     '▁ré'     'pl'     'ique'     '▁suiv'     'ante'     '.'     '<0x0A>'     '<0x0A>'     'BO'     'H'     'ORT'     ':'     '▁La'     '▁fem'     'elle'     '▁li'     'è'     'vre'     ','     '▁c'     '’'     'est'     '▁la'     '▁h'     'ase'     '.'     '[/INST]'   ✔ '▁K'
 ✔ 'AR'
 ✔ 'AD'
 ✔ 'OC'
 ✔ ':'
 ✔ '▁Ah'
 ✔ '▁non'
 ✔ ','
 ✔ '▁mo'
 ✔ 'i'
 ✔ '▁je'
 ✔ '▁con'
 ✔ 'na'
 ✔ 'is'
 ✔ '▁que'
 ✔ '▁le'
 ✔ '▁cri'
 ✔ '.'
 ✔ '</s>'


Les tokens ✔ sont la réponse, **y compris `</s>`** : le modèle apprend aussi **quand s'arrêter**. Les bibliothèques font ce décalage pour nous (TRL : `completion_only_loss=True`, notebook 05), mais c'est exactement ce calcul.

> **Décalage d'un cran.** Au chapitre 2, `y` était `x` décalé d'un caractère. Avec Hugging Face, on passe `labels` **alignés** sur `input_ids`, et le modèle fait le décalage en interne : la position `t` prédit le label `t+1`.

---
## 5. Un corpus, trois objectifs

Le **même** corpus peut servir trois tâches différentes. Ce n'est pas l'architecture qui change, c'est **la construction des couples (entrée, cible)** :

In [8]:
scenes = ft.lire_dialogues("kaamelott")
scene = scenes[0]

print("=== 1. Adaptation de domaine : texte brut, loss sur tout ===")
print("\n".join(ft.formater_replique(r) for r in scene[:4]))

print("\n=== 2. Style conversationnel : contexte -> réplique suivante ===")
ex = ft.construire_exemples([scene], "kaamelott", k=3, mode="suite")[2]
print("PROMPT  :", ex["prompt"][0]["content"])
print("RÉPONSE :", ex["completion"][0]["content"])

print("\n=== 3. Instruction : « réponds comme tel personnage » ===")
ex = ft.construire_exemples([scene], "kaamelott", k=3, mode="personnage")[2]
print("PROMPT  :", ex["prompt"][0]["content"])
print("RÉPONSE :", ex["completion"][0]["content"])

=== 1. Adaptation de domaine : texte brut, loss sur tout ===
KARADOC: Bon, je vais essayer de trouver un petit lièvre pour ce soir parce qu’il commence à faire faim !
BOHORT: Je me demande comment vous faîtes, moi je serais incapable de trouver de quoi manger dans la forêt.
KARADOC: Pour attraper des bêtes, il faut imiter les femelles. Là, pour le coup, hop ! La femelle lièvre.
BOHORT: La hase.

=== 2. Style conversationnel : contexte -> réplique suivante ===
PROMPT  : Continue ce dialogue de Kaamelott avec la réplique suivante.

KARADOC: Bon, je vais essayer de trouver un petit lièvre pour ce soir parce qu’il commence à faire faim !
BOHORT: Je me demande comment vous faîtes, moi je serais incapable de trouver de quoi manger dans la forêt.
KARADOC: Pour attraper des bêtes, il faut imiter les femelles. Là, pour le coup, hop ! La femelle lièvre.
RÉPONSE : BOHORT: La hase.

=== 3. Instruction : « réponds comme tel personnage » ===
PROMPT  : Réponds comme BOHORT dans Kaamelott.

KARADOC: B

| Objectif | Ce que le modèle apprend | Usage |
|---|---|---|
| 1. Adaptation de domaine | la distribution du texte (vocabulaire, tournures) | continuer du texte ; c'est ce que faisait le chapitre 2 |
| 2. Style conversationnel | répondre à un contexte **et** choisir qui parle | générer la suite d'une scène |
| 3. Instruction | obéir à une consigne explicite (« réponds comme Perceval ») | chatbot « dans le rôle » |

Dans ce chapitre, nous utilisons l'**objectif 2** (le fichier exporté au notebook 02). Le devoir propose de comparer avec l'objectif 3.

---
## 6. Ce qui peut mal tourner

- **Oubli catastrophique** : en poussant les poids vers Kaamelott, on peut abîmer ce que le modèle savait faire (le français soutenu, répondre à une question). On le **mesurera** au notebook 03.
- **Mémorisation** : avec peu de données (Kaamelott !), le modèle peut réciter le corpus au lieu de le généraliser, exactement comme au chapitre 2.
- **Perte de l'alignement** : un modèle Instruct a appris à refuser certaines demandes. Un fine-tuning de style peut éroder ce comportement : l'évaluation (notebook 06) doit inclure **ce que l'on ne voulait pas changer**.
- **Mémoire** : 7 milliards de paramètres ne tiennent pas facilement sur un GPU. Notebooks 03 et 04 : LoRA et quantification 4 bits.

## À retenir

- Mistral 7B **est** un GPT decoder-only : embeddings, blocs attention + FeedForward, résidus, normalisation, `lm_head`. Seules l'échelle et quelques briques (RoPE, GQA, SwiGLU, RMSNorm) changent.
- Le tokeniseur BPE découpe en **sous-mots** : ≈ 3 caractères par token, vocabulaire de 32 768.
- Fine-tuner = **continuer l'entraînement** avec la même cross-entropy, **masquée** (`-100`) sur le prompt.
- C'est la **construction des exemples** qui définit la tâche apprise.

**Questions de vérification**
1. Pourquoi `k_proj` est-il 4 fois plus petit que `q_proj` dans Mistral ?
2. Que se passerait-il si on ne masquait pas le prompt dans la loss ?
3. Pour un assistant qui répond sur les horaires de la bibliothèque, choisiriez-vous RAG ou fine-tuning ? Pourquoi ?
4. Pourquoi le token `</s>` doit-il faire partie de la réponse dans la loss ?